# Path Divergence Metrics

This notebook computes metrics for comparing paths between two markers across different elevation maps.
We place markers at fixed positions (0.2 × resolution, 0.8 × resolution) and compute paths for different
elevation samples. The main metric is the area between paths divided by the bounding box area.

In [ ]:
import json
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

# Add src to path for imports
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
from passage.generate import find_path, load_tile_grid, _resolve_solver_config
from passage.visualization import tensor_to_image
from scipy.spatial.distance import directed_hausdorff

In [ ]:
# Configuration
CONFIG_PATH = ROOT / "config" / "config.yaml"
TILES_DIR = ROOT / "outputs" / "download" / "tiles"
CALIBRATE_PATH = ROOT / "outputs" / "calibrate" / "calibrate.json"

RESOLUTION = 1024  # Resolution of elevation maps
DEFAULT_COST_MODEL = "elevation"

# Load config
try:
    with open(CONFIG_PATH) as f:
        _config = yaml.safe_load(f) or {}
    _solver_cfg = _config.get("solver", {})
    COST_MODEL = (
        _solver_cfg.get("grid", {}).get("cost_model")
        or _solver_cfg.get("networkx", {}).get("cost_model")
        or _solver_cfg.get("pathfinding", {}).get("cost_model")
        or DEFAULT_COST_MODEL
    )
except Exception:
    COST_MODEL = DEFAULT_COST_MODEL

# Solver config for path finding
SOLVER_CONFIG = _resolve_solver_config(_solver_cfg)

print("Solver Config:\n" + json.dumps(SOLVER_CONFIG, indent=2))

In [ ]:
# Load calibration data
if not CALIBRATE_PATH.exists():
    raise FileNotFoundError(f"Calibration file not found: {CALIBRATE_PATH}. Run 'passage calibrate' first.")
if not TILES_DIR.exists():
    raise FileNotFoundError(f"Tiles directory not found: {TILES_DIR}. Run 'passage download' first.")

with open(CALIBRATE_PATH) as f:
    calibration = json.load(f)

global_min = float(calibration["global_min"])
global_max = float(calibration["global_max"])
tile_stats_df = pd.DataFrame(calibration["tile_stats"])

print(f"Global elevation range: [{global_min:.1f}, {global_max:.1f}]")
print(f"Number of tiles: {len(tile_stats_df)}")

## Helper Functions

Functions to generate elevation crops and compute paths.

In [ ]:
import math
import random

TILE_SIZE_PX = 3600


def generate_random_crop(
    resolution: int,
    tile_stats_df: pd.DataFrame,
    tiles_dirpath: Path,
    global_min: float,
    global_max: float,
    seed: int | None = None,
) -> np.ndarray | None:
    """Generate a random elevation crop of the given resolution.

    Returns:
        Elevation crop as numpy array, or None if generation failed.
    """
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    half_res = resolution // 2
    half_tiles = max(1, math.ceil(half_res / TILE_SIZE_PX))

    for _ in range(10):  # Try up to 10 times
        # Pick a random elevation
        target_elev = random.uniform(global_min, global_max)

        # Filter tiles containing this elevation
        candidates = tile_stats_df[(tile_stats_df["min"] <= target_elev) & (tile_stats_df["max"] >= target_elev)]

        if candidates.empty:
            continue

        # Select random tile
        selected = candidates.sample(n=1).iloc[0]
        center_lat = int(selected["lat"])
        center_lon = int(selected["lon"])

        # Load tile grid
        grid = load_tile_grid(tiles_dirpath, center_lat, center_lon, half_tiles=half_tiles)
        grid_h, grid_w = grid.shape

        if grid_h < resolution or grid_w < resolution:
            continue

        # Random crop position
        y1 = random.randint(0, grid_h - resolution)
        x1 = random.randint(0, grid_w - resolution)

        crop = grid[y1 : y1 + resolution, x1 : x1 + resolution]
        return crop

    return None


def get_marker_positions(resolution: int) -> tuple[tuple[int, int], tuple[int, int]]:
    """Get fixed marker positions at 0.2 and 0.8 of resolution.

    Returns:
        (start, end) marker positions as (row, col) tuples.
    """
    pos1 = int(0.2 * resolution)
    pos2 = int(0.8 * resolution)
    start = (pos1, pos1)
    end = (pos2, pos2)
    return start, end


print(f"Marker positions for resolution {RESOLUTION}: {get_marker_positions(RESOLUTION)}")

## Single Example: Two Paths on Different Elevation Maps

Generate two different elevation maps and find paths between the same marker locations.

In [ ]:
# Generate two different elevation crops
crop1 = generate_random_crop(RESOLUTION, tile_stats_df, TILES_DIR, global_min, global_max, seed=42)
crop2 = generate_random_crop(RESOLUTION, tile_stats_df, TILES_DIR, global_min, global_max, seed=123)

if crop1 is None or crop2 is None:
    raise RuntimeError("Failed to generate elevation crops")

# Get marker positions
start, end = get_marker_positions(RESOLUTION)
print(f"Start marker: {start}")
print(f"End marker: {end}")

# Find paths
path1 = find_path(crop1, start, end, SOLVER_CONFIG)
path2 = find_path(crop2, start, end, SOLVER_CONFIG)

print(f"Path 1 length: {len(path1)} pixels")
print(f"Path 2 length: {len(path2)} pixels")

In [ ]:
def draw_paths_on_black(path1: list, path2: list, resolution: int) -> np.ndarray:
    """Draw two paths on a black background.

    Args:
        path1: First path as list of (row, col) tuples
        path2: Second path as list of (row, col) tuples
        resolution: Image resolution

    Returns:
        RGB image with paths drawn
    """
    # Create black background (RGB)
    img = np.zeros((resolution, resolution, 3), dtype=np.uint8)

    # Draw path 1 in red
    if len(path1) > 0:
        path1_arr = np.array(path1)
        # OpenCV uses (x, y) = (col, row) format
        pts1 = path1_arr[:, ::-1].reshape((-1, 1, 2)).astype(np.int32)
        cv2.polylines(img, [pts1], isClosed=False, color=(255, 0, 0), thickness=2)

    # Draw path 2 in green
    if len(path2) > 0:
        path2_arr = np.array(path2)
        pts2 = path2_arr[:, ::-1].reshape((-1, 1, 2)).astype(np.int32)
        cv2.polylines(img, [pts2], isClosed=False, color=(0, 255, 0), thickness=2)

    return img


# Use tensor_to_image to visualize elevation + markers + path overlays


def make_vis_from_crop(crop, path, resolution):
    elevation_norm = (crop - global_min) / (global_max - global_min)
    elevation_norm = np.clip(elevation_norm, 0.0, 1.0).astype(np.float32)
    marker_channel = np.zeros_like(elevation_norm, dtype=np.float32)
    start_pos, end_pos = get_marker_positions(resolution)
    marker_channel[start_pos] = 1.0
    marker_channel[end_pos] = 1.0
    obstacles_channel = np.zeros_like(elevation_norm, dtype=np.float32)
    tensor_vis = np.stack([elevation_norm, marker_channel, obstacles_channel], axis=-1)
    img_bgr = tensor_to_image(tensor_vis, path_pixels=path)
    return img_bgr[:, :, ::-1]


img1 = make_vis_from_crop(crop1, path1, RESOLUTION)
img2 = make_vis_from_crop(crop2, path2, RESOLUTION)

# Both paths on black background (keep original black background visualization)
paths_img = draw_paths_on_black(path1, path2, RESOLUTION)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Elevation map 1 with path (tensor_to_image)
axes[0].imshow(img1)
axes[0].scatter([start[1], end[1]], [start[0], end[0]], c=["lime", "cyan"], s=50, zorder=5)
axes[0].set_title("Elevation Map 1 (tensor_to_image)")
axes[0].axis("off")

# Elevation map 2 with path (tensor_to_image)
axes[1].imshow(img2)
axes[1].scatter([start[1], end[1]], [start[0], end[0]], c=["lime", "cyan"], s=50, zorder=5)
axes[1].set_title("Elevation Map 2 (tensor_to_image)")
axes[1].axis("off")

# Both paths on black background
axes[2].imshow(paths_img)
axes[2].set_title("Both Paths on Black Background")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## Compute Area Between Paths

Use OpenCV to compute the area enclosed between the two paths.

In [ ]:
def compute_area_between_paths(path1: list, path2: list, resolution: int) -> tuple[int, np.ndarray]:
    """Compute the area (in pixels) between two paths using OpenCV.

    The area is computed by:
    1. Creating a closed polygon from path1 -> reversed(path2)
    2. Filling the polygon and counting pixels

    Args:
        path1: First path as list of (row, col) tuples
        path2: Second path as list of (row, col) tuples
        resolution: Image resolution

    Returns:
        (area_pixels, mask) where mask is the filled area visualization
    """
    if len(path1) == 0 or len(path2) == 0:
        return 0, np.zeros((resolution, resolution), dtype=np.uint8)

    # Create a closed polygon: path1 followed by reversed path2
    path1_arr = np.array(path1)
    path2_arr = np.array(path2)

    # Combine paths to form a closed contour
    # path1: start -> end, path2 reversed: end -> start
    combined = np.vstack([path1_arr, path2_arr[::-1]])

    # Convert to OpenCV format (col, row)
    pts = combined[:, ::-1].astype(np.int32)

    # Create mask and fill polygon
    mask = np.zeros((resolution, resolution), dtype=np.uint8)
    cv2.fillPoly(mask, [pts], color=255)

    # Count pixels
    area_pixels = np.sum(mask > 0)

    return area_pixels, mask


# Compute area between the two paths
area, area_mask = compute_area_between_paths(path1, path2, RESOLUTION)
print(f"Area between paths: {area} pixels")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(paths_img)
axes[0].set_title("Paths")
axes[0].axis("off")

axes[1].imshow(area_mask, cmap="hot")
axes[1].set_title(f"Area Between Paths: {area} pixels")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## Path Divergence Metric Function

Compute the area between paths and normalize by the bounding box area.

In [ ]:
def compute_path_divergence_metric(path1: list, path2: list, resolution: int) -> dict:
    """Compute path divergence metric: area between paths / bounding box area.

    Args:
        path1: First path as list of (row, col) tuples
        path2: Second path as list of (row, col) tuples
        resolution: Image resolution

    Returns:
        Dictionary with metrics:
            - area_pixels: Number of pixels between paths
            - bbox_area: Bounding box area in pixels
            - bbox: (min_row, min_col, max_row, max_col)
            - divergence_ratio: area_pixels / bbox_area
    """
    if len(path1) == 0 or len(path2) == 0:
        return {
            "area_pixels": 0,
            "bbox_area": 0,
            "bbox": (0, 0, 0, 0),
            "divergence_ratio": 0.0,
        }

    # Compute area between paths
    area_pixels, _ = compute_area_between_paths(path1, path2, resolution)

    # Compute bounding box encompassing both paths
    path1_arr = np.array(path1)
    path2_arr = np.array(path2)
    all_points = np.vstack([path1_arr, path2_arr])

    min_row = int(all_points[:, 0].min())
    max_row = int(all_points[:, 0].max())
    min_col = int(all_points[:, 1].min())
    max_col = int(all_points[:, 1].max())

    bbox_height = max_row - min_row + 1
    bbox_width = max_col - min_col + 1
    bbox_area = bbox_height * bbox_width

    # Compute divergence ratio
    divergence_ratio = area_pixels / bbox_area if bbox_area > 0 else 0.0

    return {
        "area_pixels": area_pixels,
        "bbox_area": bbox_area,
        "bbox": (min_row, min_col, max_row, max_col),
        "divergence_ratio": divergence_ratio,
    }


# Test the function
metrics = compute_path_divergence_metric(path1, path2, RESOLUTION)
print(f"Area between paths: {metrics['area_pixels']} pixels")
print(f"Bounding box area: {metrics['bbox_area']} pixels")
print(f"Bounding box: {metrics['bbox']}")
print(f"Divergence ratio: {metrics['divergence_ratio']:.4f}")

## Additional Path Comparison Metrics

We define several normalized metrics to compare paths. All metrics are in [0, 1].

### Metrics Summary

| Metric | Description | Range | Interpretation |
|--------|-------------|-------|----------------|
| **divergence_ratio** | Area between paths / bounding box area | [0, 1] | 0 = identical |
| **normalized_hausdorff** | Max point-to-set distance / diagonal | [0, 1] | 0 = identical |
| **length_diff_ratio** | Relative path length difference | [0, 1] | 0 = same length |
| **mean_deviation** | Average mean point-to-path distances | [0, 1] | 0 = identical |
| **overlap_error** | 1 - Jaccard index of buffered paths | [0, 1] | 0 = perfect overlap |
| **overlap_iou** | Jaccard index (IoU) of buffered paths | [0, 1] | 1 = perfect overlap |
| **divergence_hausdorff_mix** | (divergence_ratio + normalized_hausdorff) / 2 | [0, 1] | 0 = identical |

### How Metrics Are Computed

1. **divergence_ratio**: Creates polygon from both paths, computes `filled_area / bbox_area`
2. **normalized_hausdorff**: `max(d(A→B), d(B→A)) / (resolution × √2)` using scipy
3. **length_diff_ratio**: `|len(path1) - len(path2)| / max(len1, len2)`
4. **mean_deviation**: Average of mean min-distances in both directions / diagonal
5. **overlap_error**: Binary masks + dilation (r=2), then `1 - (intersection / union)`
6. **overlap_iou**: Same as above but returns IoU directly (1 = perfect match)
7. **divergence_hausdorff_mix**: `(divergence_ratio + normalized_hausdorff) / 2`


In [ ]:
def hausdorff_distance(path1: list, path2: list) -> float:
    """Compute Hausdorff distance between two paths using scipy."""
    if len(path1) == 0 or len(path2) == 0:
        return 0.0
    arr1 = np.array(path1)
    arr2 = np.array(path2)
    d1 = directed_hausdorff(arr1, arr2)[0]
    d2 = directed_hausdorff(arr2, arr1)[0]
    return max(d1, d2)


def normalized_hausdorff_distance(path1: list, path2: list) -> float:
    """Compute normalized Hausdorff distance in [0, 1]."""
    haus = hausdorff_distance(path1, path2)
    dist = np.linalg.norm(np.array(path1[-1]) - np.array(path1[0]))
    return haus / max(dist, 1e-6)


def normalized_area_between_paths(path1: list, path2: list, resolution: int) -> float:
    area_pixels, _ = compute_area_between_paths(path1, path2, resolution)
    dist = np.linalg.norm(np.array(path1[-1]) - np.array(path1[0]))
    return area_pixels / max(dist**2, 1e-6)


def hausdorff_area_metric_0(path1: list, path2: list, resolution: int) -> float:
    """Compute combined Hausdorff + area metric in [0, 1]."""
    norm_haus = normalized_hausdorff_distance(path1, path2)
    norm_area = normalized_area_between_paths(path1, path2, resolution)
    return norm_haus / (1 + norm_area)


def hausdorff_area_metric_1(path1: list, path2: list, resolution: int) -> float:
    """Compute combined Hausdorff + area metric in [0, 1]."""
    norm_haus = normalized_hausdorff_distance(path1, path2)
    norm_area = normalized_area_between_paths(path1, path2, resolution)
    return norm_area / (1 + norm_haus)


def hausdorff_area_metric_2(path1: list, path2: list, resolution: int) -> float:
    """Compute combined Hausdorff + area metric in [0, 1]."""
    norm_haus = normalized_hausdorff_distance(path1, path2)
    norm_area = normalized_area_between_paths(path1, path2, resolution)
    return 1 / (1 + norm_area * norm_haus)


def mean_deviation(path1: list, path2: list, resolution: int) -> float:
    """Compute average mean point-to-path distance in [0, 1]."""
    if len(path1) == 0 or len(path2) == 0:
        return 0.0
    arr1 = np.array(path1)
    arr2 = np.array(path2)
    dists1 = np.array([np.min(np.sqrt(np.sum((arr2 - p) ** 2, axis=1))) for p in arr1])
    dists2 = np.array([np.min(np.sqrt(np.sum((arr1 - p) ** 2, axis=1))) for p in arr2])
    dist = np.linalg.norm(np.array(path1[-1]) - np.array(path1[0]))
    return min((np.mean(dists1) + np.mean(dists2)) / (2 * max(dist, 1e-6)), 1.0)


def _create_path_masks(path1, path2, resolution, buffer_radius=2):
    """Helper to create dilated binary masks for both paths."""
    mask1 = np.zeros((resolution, resolution), dtype=np.uint8)
    mask2 = np.zeros((resolution, resolution), dtype=np.uint8)
    for r, c in path1:
        if 0 <= r < resolution and 0 <= c < resolution:
            mask1[r, c] = 1
    for r, c in path2:
        if 0 <= r < resolution and 0 <= c < resolution:
            mask2[r, c] = 1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * buffer_radius + 1, 2 * buffer_radius + 1))
    return cv2.dilate(mask1, kernel), cv2.dilate(mask2, kernel)


def compute_all_metrics(path1: list, path2: list, resolution: int) -> dict:
    """Compute all 7 path comparison metrics. Returns dict with metrics in [0, 1]."""
    div_metrics = compute_path_divergence_metric(path1, path2, resolution)
    return {
        "divergence_ratio": div_metrics["divergence_ratio"],
        # "hausdorff_distance": hausdorff_distance(path1, path2),
        "normalized_hausdorff_distance": normalized_hausdorff_distance(path1, path2),
        "normalized_area_between_paths": normalized_area_between_paths(path1, path2, resolution),
        "hausdorff_area_metric_0": hausdorff_area_metric_0(path1, path2, resolution),
        "hausdorff_area_metric_1": hausdorff_area_metric_1(path1, path2, resolution),
        "hausdorff_area_metric_2": hausdorff_area_metric_2(path1, path2, resolution),
        "mean_deviation": mean_deviation(path1, path2, resolution),
    }


# Test all metrics
all_metrics = compute_all_metrics(path1, path2, RESOLUTION)
print("All Path Comparison Metrics:")
for name, value in all_metrics.items():
    print(f"  {name}: {value:.4f}")

## Generate 20 Path Pair Cases

Generate 20 cases with different elevation maps and compute the divergence metric for each.

In [ ]:
NUM_CASES = 40

results = []

for i in range(NUM_CASES):
    seed1 = i * 1000
    seed2 = i * 1000 + 500

    crop1 = generate_random_crop(RESOLUTION, tile_stats_df, TILES_DIR, global_min, global_max, seed=seed1)
    crop2 = generate_random_crop(RESOLUTION, tile_stats_df, TILES_DIR, global_min, global_max, seed=seed2)

    if crop1 is None or crop2 is None:
        print(f"Case {i + 1}: Failed to generate crops, skipping")
        continue

    start, end = get_marker_positions(RESOLUTION)
    path1 = find_path(crop1, start, end, SOLVER_CONFIG)
    path2 = find_path(crop2, start, end, SOLVER_CONFIG)

    if len(path1) == 0 or len(path2) == 0:
        print(f"Case {i + 1}: No path found, skipping")
        continue

    all_metrics = compute_all_metrics(path1, path2, RESOLUTION)

    results.append(
        {
            "case": i + 1,
            "path1_length": len(path1),
            "path2_length": len(path2),
            **all_metrics,
        }
    )

print(f"\nSuccessfully processed {len(results)}/{NUM_CASES} cases")

In [ ]:
# Create DataFrame with results
results_df = pd.DataFrame(results)
results_df

In [ ]:
# Summary statistics for all metrics
metric_cols = list(all_metrics.keys())

print("Summary Statistics for All Normalized Metrics:")
print("=" * 60)
for col in metric_cols:
    print(f"\n{col}:")
    print(f"  Mean:   {results_df[col].mean():.4f}")
    print(f"  Std:    {results_df[col].std():.4f}")
    print(f"  Min:    {results_df[col].min():.4f}")
    print(f"  Max:    {results_df[col].max():.4f}")
    print(f"  Median: {results_df[col].median():.4f}")

In [ ]:
# Histograms of all metrics
metric_cols = list(all_metrics.keys())

for idx, (col, label) in enumerate(zip(metric_cols, metric_cols)):
    fig = plt.figure(figsize=(6, 4))
    ax = fig.add_subplot(1, 1, 1)
    bins = np.arange(0.0, 1.0 + 0.05, 0.05)
    ax.hist(results_df[col], bins=bins, edgecolor="black", alpha=0.7, color="steelblue")
    ax.axvline(x=results_df[col].mean(), color="red", linestyle="--", label="Mean")
    ax.set_xlabel(label)
    ax.set_ylabel("Frequency")
    ax.set_title(f"Distribution of {label}")
    ax.set_xlim(0, 1)
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation heatmap of all metrics (matplotlib only, no seaborn)
metric_cols = list(all_metrics.keys())
corr_matrix = results_df[metric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)

cbar = ax.figure.colorbar(im, ax=ax)
cbar.ax.set_ylabel("Correlation", rotation=-90, va="bottom")

ax.set_xticks(range(len(metric_cols)))
ax.set_yticks(range(len(metric_cols)))
ax.set_xticklabels(metric_cols, rotation=45, ha="right")
ax.set_yticklabels(metric_cols)

for i in range(len(metric_cols)):
    for j in range(len(metric_cols)):
        ax.text(j, i, f"{corr_matrix.iloc[i, j]:.2f}", ha="center", va="center", color="black", fontsize=9)

ax.set_title("Correlation Between Path Comparison Metrics")
plt.tight_layout()
plt.show()

## Visualize Sample Cases

Show a few example cases with their paths and divergence metrics.

In [ ]:
# Show all cases with paths and metric bar charts
sample_cases = range(len(results))

fig, axes = plt.subplots(len(sample_cases), 3, figsize=(15, 5 * len(sample_cases)))

metric_names = list(all_metrics.keys())
metric_labels = [str(i + 1) for i in range(len(metric_names))]

print("metric_labels:", metric_labels)
print("name:", name)

for idx, case_idx in enumerate(sample_cases):
    seed1 = case_idx * 1000
    seed2 = case_idx * 1000 + 500

    crop1 = generate_random_crop(RESOLUTION, tile_stats_df, TILES_DIR, global_min, global_max, seed=seed1)
    crop2 = generate_random_crop(RESOLUTION, tile_stats_df, TILES_DIR, global_min, global_max, seed=seed2)

    if crop1 is None or crop2 is None:
        continue

    start, end = get_marker_positions(RESOLUTION)
    path1 = find_path(crop1, start, end, SOLVER_CONFIG)
    path2 = find_path(crop2, start, end, SOLVER_CONFIG)

    all_metrics = compute_all_metrics(path1, path2, RESOLUTION)
    paths_img = draw_paths_on_black(path1, path2, RESOLUTION)

    ax_row = axes[idx] if len(sample_cases) > 1 else axes

    # Path 1 on elevation
    elevation_norm1 = np.clip((crop1 - global_min) / (global_max - global_min), 0.0, 1.0).astype(np.float32)
    marker_channel = np.zeros_like(elevation_norm1, dtype=np.float32)
    marker_channel[start] = 1.0
    marker_channel[end] = 1.0
    tensor_vis1 = np.stack([elevation_norm1, marker_channel, np.zeros_like(elevation_norm1)], axis=-1)
    img_bgr1 = tensor_to_image(tensor_vis1, path_pixels=path1)
    img1 = img_bgr1[:, :, ::-1]
    ax_row[0].imshow(img1)
    ax_row[0].scatter([start[1], end[1]], [start[0], end[0]], c=["lime", "cyan"], s=40, zorder=5)
    ax_row[0].set_title(f"Case {case_idx + 1}: Elevation 1")
    ax_row[0].axis("off")

    # Both paths
    ax_row[1].imshow(paths_img)
    ax_row[1].set_title("Both Paths (Blue=Path1, Red=Path2)")
    ax_row[1].axis("off")

    # Bar chart of all metrics
    metric_values = [all_metrics[m] for m in metric_names]
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(metric_names)))
    bars = ax_row[2].bar(metric_labels, metric_values, color=colors)
    ax_row[2].set_ylim(0, 1)
    ax_row[2].set_ylabel("Value [0-1]")
    ax_row[2].set_title("Metrics (0=identical, IoU: 1=identical)")
    ax_row[2].tick_params(axis="x", rotation=45)
    for bar, val in zip(bars, metric_values):
        ax_row[2].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f"{val:.2f}",
            ha="center",
            va="bottom",
            fontsize=8,
        )

plt.tight_layout()
plt.show()